#  AnimeGPT — Fine-tuning GPT-2 on Anime Dialogue

This notebook fine-tunes **GPT-2** on anime subtitle dialogue so it learns to generate
contextually relevant anime-style responses — not just random character-level text.

**This notebook is fully self-contained.** Run every cell top to bottom — do not mix in
code from any older notebook (e.g. a from-scratch character-level transformer). 

**Requirements:**
- GPU with 6GB+ VRAM (RTX 3050 works)
- Python 3.10–3.12 (PyTorch does not yet support 3.13/3.14)
- Your subtitle files in `context: ... response: ...` format, all in one folder

## Step 0 — Sanity Check (Python + GPU)

Run this first. If `torch` fails to import or CUDA shows `False`, stop and fix your
environment before continuing — training will silently run on CPU and take days.

In [1]:
import sys
print(f"Python version: {sys.version}")

import torch
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be extremely slow on CPU.")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## Step 1 — Install Dependencies

In [2]:
!pip install transformers datasets accelerate -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Config

Edit `DATASET_DIR` to point at your folder of subtitle files.

**Speed settings below are tuned to finish in ~2–3 hours on an RTX 3050.**
If your first run took 30+ hours, it's because `MAX_LENGTH` and/or `MAX_FILES` were too
high — the defaults here fix that. Increase them later once you confirm everything works.

In [3]:
import os

# ===== PATHS =====
# Use a raw string (r"...") or forward slashes so Windows backslashes don't break.
DATASET_DIR   = r"C:\Users\nipan\OneDrive\Desktop\AnimeGPT\Dataset"   # <-- folder with your train-anime-*.txt files
OUTPUT_DIR    = "animegpt-model"      # fine-tuned model saved here
CORPUS_FILE   = "anime_corpus.txt"    # auto-generated preprocessed corpus

# ===== MODEL =====
MODEL_NAME    = "gpt2"   # ~124M params. Use "gpt2-medium" only if you have more VRAM/time.

# ===== SPEED / SIZE (tuned for RTX 3050 6GB, ~2-3hr run) =====
MAX_FILES     = 500      # number of subtitle files to use. Set to None to use ALL files (slow).
MAX_LENGTH    = 128      # token context window. 256 doubles time/memory — keep at 128 for first run.
BATCH_SIZE    = 4        # per-step batch size, safe for 6GB VRAM
GRAD_ACCUM    = 8        # effective batch = BATCH_SIZE * GRAD_ACCUM = 32
EPOCHS        = 2        # 2 is enough to see good results; 3+ takes much longer
LR            = 5e-5
WARMUP_STEPS  = 200
SAVE_STEPS    = 500

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Dataset folder exists: {os.path.isdir(DATASET_DIR)}")
if os.path.isdir(DATASET_DIR):
    n_files = len([f for f in os.listdir(DATASET_DIR) if f.endswith(".txt")])
    print(f"Found {n_files} .txt files in dataset folder")

Device: cuda
Dataset folder exists: True
Found 3836 .txt files in dataset folder


TO delete old corpus for re running code.

In [4]:
import os
if os.path.exists("anime_corpus.txt"):
    os.remove("anime_corpus.txt")
    print("Old corpus deleted — ready to rebuild")
else:
    print("No existing corpus found")

Old corpus deleted — ready to rebuild


## Step 3 — Preprocess Dataset

Each line in your files looks like:
```
context: Line1\Line2\Line3 response:\tNext line
```
We convert this into a structured prompt GPT-2 learns from:
```
<|context|>
Line1
Line2
Line3
<|response|>
Next line
<|endoftext|>
```

In [5]:
import re
import random
from pathlib import Path


def parse_file(filepath):
    """Extract (context, response) pairs from a single subtitle file."""
    pairs = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = re.match(r"context:\s+(.+?)\s+response:\t(.+)$", line)
            if not match:
                continue
            ctx_raw, response = match.group(1), match.group(2).strip()
            if not response or len(response) < 2:
                continue
            ctx_lines = [l.strip() for l in ctx_raw.split("\\") if l.strip()]
            ctx_lines = ctx_lines[-6:]   # keep last 6 lines of context
            pairs.append((ctx_lines, response))
    return pairs


def format_pair(ctx_lines, response):
    context_str = "\n".join(ctx_lines)
    return f"<|context|>\n{context_str}\n<|response|>\n{response}\n<|endoftext|>"


def build_corpus(dataset_dir, corpus_file, max_files=None):
    if os.path.exists(corpus_file):
        print(f"'{corpus_file}' already exists — delete it first if you want to rebuild.")
        return

    files = sorted(Path(dataset_dir).glob("*.txt"))
    if max_files:
        files = files[:max_files]
    print(f"Processing {len(files)} files...")

    all_pairs = []
    for fp in files:
        all_pairs.extend(parse_file(fp))

    random.seed(42)
    random.shuffle(all_pairs)

    seen = set()
    unique_pairs = []
    for ctx, resp in all_pairs:
        if resp not in seen:
            seen.add(resp)
            unique_pairs.append((ctx, resp))

    print(f"Total pairs: {len(all_pairs):,} → after dedup: {len(unique_pairs):,}")

    with open(corpus_file, "w", encoding="utf-8") as f:
        for ctx, resp in unique_pairs:
            f.write(format_pair(ctx, resp) + "\n")

    size_mb = os.path.getsize(corpus_file) / 1e6
    print(f"Corpus saved: '{corpus_file}' ({size_mb:.1f} MB)")


build_corpus(DATASET_DIR, CORPUS_FILE, max_files=MAX_FILES)

Processing 500 files...
Total pairs: 190,099 → after dedup: 145,192
Corpus saved: 'anime_corpus.txt' (41.4 MB)


## Step 4 — Tokenizer (with special dialogue tokens)

In [6]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

special_tokens = {
    "pad_token": "<|pad|>",
    "additional_special_tokens": ["<|context|>", "<|response|>"]
}
tokenizer.add_special_tokens(special_tokens)

print(f"Vocab size: {len(tokenizer):,}")
print(f"Special tokens: {tokenizer.all_special_tokens}")

test = "<|context|>\nAre you okay?\n<|response|>\nI'm fine, don't worry.\n<|endoftext|>"
tokens = tokenizer.encode(test)
print(f"\nSample encoded ({len(tokens)} tokens): {tokens[:10]}...")
print(f"Decoded back: {tokenizer.decode(tokens[:15])}...")

c:\Users\nipan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vocab size: 50,260
Special tokens: ['<|endoftext|>', '<|pad|>', '<|context|>', '<|response|>']

Sample encoded (19 tokens): [50258, 198, 8491, 345, 8788, 30, 198, 50259, 198, 40]...
Decoded back: <|context|>
Are you okay?
<|response|>
I'm fine, don't...


## Step 5 — Tokenize Dataset

In [7]:
from datasets import Dataset

def load_and_tokenize(corpus_file, tokenizer, max_length, val_split=0.05):
    with open(corpus_file, "r", encoding="utf-8") as f:
        raw_text = f.read()

    examples = [e.strip() for e in raw_text.split("<|endoftext|>") if e.strip()]
    print(f"Total examples: {len(examples):,}")

    def tokenize(batch):
        return tokenizer(
            [text + "<|endoftext|>" for text in batch["text"]],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )

    dataset = Dataset.from_dict({"text": examples})
    dataset = dataset.map(
        tokenize, batched=True, batch_size=1000,
        remove_columns=["text"], desc="Tokenizing"
    )
    dataset.set_format(type="torch")

    split = dataset.train_test_split(test_size=val_split, seed=42)
    print(f"Train: {len(split['train']):,} | Val: {len(split['test']):,}")
    return split["train"], split["test"]


train_dataset, val_dataset = load_and_tokenize(CORPUS_FILE, tokenizer, MAX_LENGTH)

Total examples: 145,192


Tokenizing: 100%|██████████| 145192/145192 [00:07<00:00, 20057.50 examples/s]

Train: 137,932 | Val: 7,260


## Step 6 — Load Model

In [8]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {total_params/1e6:.1f}M")
print(f"Device: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4354.36it/s]
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model: gpt2
Parameters: 124.4M
Device: cuda:0


## Step 7 — Train

Note: this uses `eval_strategy` (not `evaluation_strategy` — that argument was renamed
in newer `transformers` versions and will throw a `TypeError` if used).

In [9]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # GPT-2 is causal LM, not masked LM
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    fp16=True,                    
    eval_strategy="steps",         
    eval_steps=SAVE_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none",
    dataloader_num_workers=0,
    max_grad_norm=1.0,
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Starting training...")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
trainer.train()

Starting training...
Effective batch size: 32


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,3.058340,2.953544
1000,2.925510,2.859760
1500,2.884421,2.785805
2000,2.835449,2.735915
2500,2.771816,2.690941
3000,2.747982,2.652657
3500,2.720290,2.620808
4000,2.673405,2.590988
4500,2.605641,2.567497
5000,2.597039,2.544488


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.66it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=8622, training_loss=2.70653658172034, metrics={'train_runtime': 8999.7368, 'train_samples_per_second': 30.652, 'train_steps_per_second': 0.958, 'total_flos': 1.8020266278912e+16, 'train_loss': 2.70653658172034, 'epoch': 2.0})

## Step 8 — Save Model

In [10]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to '{OUTPUT_DIR}/'")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]

Model saved to 'animegpt-model/'


## Step 9 — Generate Anime Dialogue

Give the model a conversation history and it generates the next line. 

In [2]:
def generate_response(context_lines, model, tokenizer,
                       max_new_tokens=80,
                       temperature=0.85,
                       top_k=50,
                       top_p=0.92,
                       num_responses=3):
    model.eval()

    context_str = "\n".join(context_lines[-6:])
    prompt = f"<|context|>\n{context_str}\n<|response|>\n"

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            num_return_sequences=num_responses,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    responses = []
    for output in outputs:
        generated = output[input_ids.shape[-1]:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        text = text.split("\n")[0].strip()
        if text:
            responses.append(text)

    return responses


context = [
    "Are you really going to fight him alone?",
    "You'll get yourself killed!",
    "I don't care. This is my fight.",
]

print("=" * 50)
print("CONTEXT:")
for line in context:
    print(f"  {line}")
print("\nANIMEGPT RESPONSES:")
responses = generate_response(context, model, tokenizer)
for i, r in enumerate(responses, 1):
    print(f"  [{i}] {r}")

CONTEXT:
  Are you really going to fight him alone?
  You'll get yourself killed!
  I don't care. This is my fight.

ANIMEGPT RESPONSES:
  [1] You've done well to earn your trust.
  [2] I don't need to run away!
  [3] All right! Get out of here!


## Step 10 — Interactive Chat Loop

In [ ]:
def chat(model, tokenizer, turns=10):
    print("AnimeGPT Chat — type 'quit' to exit, 'reset' to clear history")
    print("-" * 50)
    history = []

    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() == "quit":
            print("Sayonara!")
            break
        if user_input.lower() == "reset":
            history = []
            print("[History cleared]")
            continue

        history.append(user_input)
        responses = generate_response(history, model, tokenizer, num_responses=1)
        if responses:
            reply = responses[0]
            print(f"AnimeGPT: {reply}")
            history.append(reply)
        else:
            print("AnimeGPT: ...")

        if len(history) > 12:
            history = history[-12:]


chat(model, tokenizer)

AnimeGPT Chat — type 'quit' to exit, 'reset' to clear history
--------------------------------------------------


## Step 11 — Reload Saved Model (skip training next time)

Run this cell instead of Steps 1–8 once you already have a trained model saved.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

OUTPUT_DIR = "animegpt-model"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = GPT2Tokenizer.from_pretrained(OUTPUT_DIR)
model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).to(device)
model.eval()

print(f"Loaded AnimeGPT from '{OUTPUT_DIR}'")
 print(f"Vocab size: {len(tokenizer):,}")

c:\Users\nipan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5800.22it/s]


Loaded AnimeGPT from 'animegpt-model'
Vocab size: 50,260
